# Python Basics: Text and CSV / Bài tập Python xử lý văn bản và CSV

**Input:** các dòng văn bản hoặc file CSV gồm `name,age,score`.
**Output:** chữ in hoa hoặc danh sách bộ ba đã sắp xếp và kiểm tra dữ liệu.
**Method:** functions, loops, CSV parsing and input validation.
**Run:** ba cell chạy tự động; tương tác nhập tay/upload Colab là lựa chọn mở rộng.

## Nguồn và cách chạy / Source and execution

Bài tập Python từ Colab cá nhân `25410008.ipynb`. Tôi giữ ví dụ nhập tay và
upload Colab trong các hàm tùy chọn; **Run all** chạy được với dữ liệu CSV mẫu đi kèm.

Personal coursework practice. The default examples run without interaction;
call `interactive_uppercase()` or `upload_colab_csv()` to try the optional UI.

In [1]:
def uppercase_lines(lines):
    """VI: Viết hoa từng dòng. EN: Uppercase each line."""
    return [line.upper() for line in lines]

example_lines = ["Hello world", "Python and AI"]
print("\n".join(uppercase_lines(example_lines)))

def interactive_uppercase():
    """Nhập dòng liên tiếp; Enter trên dòng trống để dừng."""
    print("Nhập từng dòng (dòng trống để kết thúc):")
    lines = []
    while True:
        line = input()
        if not line:
            break
        lines.append(line)
    print("\n".join(uppercase_lines(lines)))

# Để nhập tay trong Colab / For interactive use:
# interactive_uppercase()


HELLO WORLD
PYTHON AND AI


In [2]:
import csv
import math
from io import StringIO

def read_and_sort_tuples(file_content):
    """Parse UTF-8 CSV (name,age,score), validate numbers and sort."""
    text = file_content.decode("utf-8-sig")
    records = []
    first_record = True
    reader = csv.reader(StringIO(text))
    for row_number, row in enumerate(reader, start=1):
        if not row or all(not col.strip() for col in row):
            continue
        if len(row) != 3:
            raise ValueError(f"CSV row {row_number}: expected 3 columns")
        name, age_text, score_text = (part.strip() for part in row)
        if first_record and [name.lower(), age_text.lower(), score_text.lower()] == ["name", "age", "score"]:
            first_record = False
            continue
        first_record = False
        if not name:
            raise ValueError(f"CSV row {row_number}: empty name")
        try:
            age, score = int(age_text), float(score_text)
        except ValueError as exc:
            raise ValueError(f"CSV row {row_number}: invalid age/score") from exc
        if not math.isfinite(score):
            raise ValueError(f"CSV row {row_number}: score must be finite")
        if age < 0:
            raise ValueError(f"CSV row {row_number}: age must be non-negative")
        records.append((name, age, score))
    return sorted(records, key=lambda r: (r[0].casefold(), r[1], r[2]))

sample_csv = b'name,age,score\nLan,20,8.5\nAn,21,9.0\nLan,19,7.5\n'
result = read_and_sort_tuples(sample_csv)
print(result)
assert [row[0] for row in result] == ["An", "Lan", "Lan"]

def upload_colab_csv():
    """Optional Google Colab upload / Tải CSV lên Colab khi cần."""
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        return []
    first_content = next(iter(uploaded.values()))
    return read_and_sort_tuples(first_content)

# For a personal file in Google Colab / Với CSV riêng:
# print(upload_colab_csv())


[('An', 21, 9.0), ('Lan', 19, 7.5), ('Lan', 20, 8.5)]


In [3]:
# Kiểm tra hành vi parser / Parser behavior checks
assert uppercase_lines(['xin chào', 'Python']) == ['XIN CHÀO', 'PYTHON']
quoted_csv = '\ufeff\nname,age,score\n"An, Nguyen",21,8.5\n'.encode('utf-8')
assert read_and_sort_tuples(quoted_csv) == [('An, Nguyen', 21, 8.5)]
invalid_rows = [b'An,-1,8', b'An,20,nan', b'An,20,inf', b'An,20', b',20,8']
for row in invalid_rows:
    try:
        read_and_sort_tuples(row)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Invalid CSV accepted: {row!r}')
LAB_RESULT = {'parser_checks': 7, 'sample_records': len(result)}
print('CSV and uppercase checks passed:', LAB_RESULT)

CSV and uppercase checks passed: {'parser_checks': 7, 'sample_records': 3}
